In [1]:
!pip install yfinance pandas numpy

In [2]:
"""
╔══════════════════════════════════════════════════════════════════╗
║          IHSG MA KETAT SCREENER — by @ikhwanuddin (adapted)     ║
║  Formula source: "Singkap Rahasia Pola MA Ketat" (Mar 2, 2026)  ║
║  Compatible: Google Colab, Jupyter Notebook, terminal           ║
╚══════════════════════════════════════════════════════════════════╝

RUMUS UTAMA:
    MA_max(t)    = max(MA3, MA5, MA10, MA20, MA50, Close)
    MA_min(t)    = min(MA3, MA5, MA10, MA20, MA50, Close)
    range_ticks  = (MA_max - MA_min) / tick_size(Close)   <- tick-adjusted BEI!
    vol_pct(t)   = std(daily_return, rolling 10hr) x 100
    ma_tight     = range_ticks < 6  AND  vol_pct < 3.8
    SINYAL FINAL = ma_tight  AND  Volume > 1.000.000  AND  MA100 <= Close
"""

# ══════════════════════════════════════════════════════════════════
# ⚙️  KONFIGURASI — ubah sesuai kebutuhan kamu
# ══════════════════════════════════════════════════════════════════
CONFIG = {
    # --- Filter MA Ketat ---
    "max_ticks"      : 6.0,       # maks range tick antar semua MA (default: 6)
    "max_vol_pct"    : 3.8,       # maks volatilitas rolling 10hr dalam % (default: 3.8)
    "max_deviation"  : 5.0,       # maks deviasi close dari setiap MA dalam % (default: 5.0)

    # --- Filter Volume Breakout ---
    "volume_filter"  : False,     # True = aktifkan filter volume
    "min_volume"     : 1_00_000, # min volume harian (lembar saham)

    # --- Filter Tren ---
    "require_above_ma100": True,  # True = Close harus >= MA100 (konteks bullish)

    # --- Saham yang di-screen ---
    # Kosongkan [] untuk pakai universe default (~120 saham IDX80)
    # Isi untuk custom, contoh: ["BBCA", "BBRI", "TLKM"]
    "custom_tickers" : [],
}
# ══════════════════════════════════════════════════════════════════


import warnings
warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "-q"])
    import yfinance as yf

import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# ══════════════════════════════════════════════════════════════════
# 1. UNIVERSE SAHAM IHSG (IDX80 + LQ45 + beberapa mid-cap aktif)
# ══════════════════════════════════════════════════════════════════
DEFAULT_TICKERS = [
    # Perbankan & Keuangan
    "BBCA", "BBRI", "BMRI", "BBNI", "BBTN", "BRIS", "BTPS", "BFIN",
    "NISP", "BJBR", "BJTM", "BNII", "PNLF",
    # Energi & Tambang
    "ADRO", "ITMG", "PTBA", "INDY", "HRUM", "MDKA", "INCO", "ANTM",
    "MEDC", "ELSA", "ESSA", "PGAS", "BRMS", "RAJA", "ADMR",
    # Konsumer
    "ICBP", "INDF", "UNVR", "MYOR", "GGRM", "KLBF", "SIDO", "RALS",
    "LPPF", "ACES", "MAPI", "AMRT", "JPFA", "CPIN", "SSMS",
    # Properti & Konstruksi
    "BSDE", "CTRA", "SMRA", "ASRI", "LPKR", "DMAS", "BKSL", "KIJA",
    "PWON", "SSIA", "ADHI", "WIKA", "PTPP", "WSKT",
    # Telko & Teknologi
    "TLKM", "EXCL", "ISAT", "TOWR", "MTEL", "TBIG", "EMTK", "MNCN",
    "GOTO", "FILM",
    # Industri & Infrastruktur
    "ASII", "UNTR", "AUTO", "JSMR", "AKRA", "TPIA", "INKP", "TKIM",
    "SMGR", "INTP", "AGII", "MARK", "ERAA",
    # Agribisnis
    "AALI", "LSIP", "TAPG",
    # Lain-lain
    "PGEO", "MIKA", "KAEF", "SRTG", "MBMA", "WMUU", "GIAA",
]
DEFAULT_TICKERS = sorted(list(dict.fromkeys(DEFAULT_TICKERS)))


# ══════════════════════════════════════════════════════════════════
# 2. TICK SIZE PIECEWISE (Aturan Resmi BEI / IDX)
# ══════════════════════════════════════════════════════════════════
def tick_size(price: float) -> int:
    """
    Fraksi harga resmi BEI (piecewise):
      < 200        -> Rp 1
      200 - 500    -> Rp 2
      500 - 2000   -> Rp 5
      2000 - 5000  -> Rp 10
      >= 5000      -> Rp 25
    """
    if price < 200:
        return 1
    elif price < 500:
        return 2
    elif price < 2000:
        return 5
    elif price < 5000:
        return 10
    else:
        return 25


def tick_size_series(prices: pd.Series) -> pd.Series:
    return prices.apply(tick_size)


# ══════════════════════════════════════════════════════════════════
# 3. FETCH DATA
# ══════════════════════════════════════════════════════════════════
def fetch_data(tickers_jk: list, period_days: int = 150) -> pd.DataFrame:
    end   = datetime.today()
    start = end - timedelta(days=period_days)
    print(f"\n📥 Mengunduh data {len(tickers_jk)} saham dari Yahoo Finance...")
    df = yf.download(
        tickers_jk,
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=True,
        threads=True,
    )
    return df


# ══════════════════════════════════════════════════════════════════
# 4. HITUNG INDIKATOR PER SAHAM
# ══════════════════════════════════════════════════════════════════
def compute_indicators(close: pd.Series, volume: pd.Series, max_dev_pct: float) -> dict:
    if len(close.dropna()) < 110:
        return None

    ma = {n: close.rolling(n).mean() for n in [3, 5, 10, 20, 50, 100]}

    # range_ticks: jarak seluruh MA dalam satuan tick (tick-adjusted BEI)
    bundle      = pd.concat([ma[3], ma[5], ma[10], ma[20], ma[50], close], axis=1)
    ma_max      = bundle.max(axis=1)
    ma_min      = bundle.min(axis=1)
    ts          = tick_size_series(close)
    range_ticks = (ma_max - ma_min) / ts

    # vol_pct: volatilitas rolling 10 hari
    daily_return = close.pct_change() * 100
    vol_pct      = daily_return.rolling(10).std()

    last = close.last_valid_index()
    if last is None:
        return None

    c       = close[last]
    vol_day = volume[last]
    rt      = range_ticks[last]
    vp      = vol_pct[last]
    ma100   = ma[100][last]

    if any(pd.isna(x) for x in [c, rt, vp, ma100]):
        return None

    # Cek close dalam +/- max_dev_pct dari SETIAP MA 3/5/10/20/50
    pct_devs = {}
    all_within_pct = True
    for n in [3, 5, 10, 20, 50]:
        ma_val = ma[n][last]
        if pd.isna(ma_val):
            all_within_pct = False
            break
        dev = abs(c - ma_val) / ma_val * 100
        pct_devs[f"dev_ma{n}%"] = round(dev, 2)
        if dev > max_dev_pct:
            all_within_pct = False

    return {
        "close"        : round(c, 0),
        "volume"       : int(vol_day),
        "ma3"          : round(ma[3][last],  0),
        "ma5"          : round(ma[5][last],  0),
        "ma10"         : round(ma[10][last], 0),
        "ma20"         : round(ma[20][last], 0),
        "ma50"         : round(ma[50][last], 0),
        "ma100"        : round(ma100,        0),
        "range_ticks"  : round(rt, 2),
        "vol_pct"      : round(vp, 2),
        "tick_size"    : tick_size(c),
        "within_pct"   : all_within_pct,
        "above_ma100"  : bool(c >= ma100),
        **pct_devs,
    }


# ══════════════════════════════════════════════════════════════════
# 5. MAIN SCREENER
# ══════════════════════════════════════════════════════════════════
def run_screener(cfg: dict) -> pd.DataFrame:
    tickers    = cfg["custom_tickers"] if cfg["custom_tickers"] else DEFAULT_TICKERS
    tickers_jk = [f"{t}.JK" for t in tickers]

    raw = fetch_data(tickers_jk)
    if raw.empty:
        print("❌ Tidak ada data yang berhasil diunduh.")
        return pd.DataFrame()

    # Support single vs multi ticker download shape
    if isinstance(raw.columns, pd.MultiIndex):
        close_all  = raw["Close"]
        volume_all = raw["Volume"]
    else:
        close_all  = raw[["Close"]].rename(columns={"Close": tickers_jk[0]})
        volume_all = raw[["Volume"]].rename(columns={"Volume": tickers_jk[0]})

    results = []
    print("🔍 Memproses indikator per saham...")

    for ticker_jk in tickers_jk:
        base = ticker_jk.replace(".JK", "")
        try:
            if ticker_jk not in close_all.columns:
                continue
            close  = close_all[ticker_jk].dropna()
            volume = volume_all[ticker_jk].reindex(close.index).fillna(0)

            ind = compute_indicators(close, volume, cfg["max_deviation"])
            if ind is None:
                continue

            ind["ticker"] = base
            results.append(ind)
        except Exception:
            pass

    if not results:
        print("⚠️  Tidak ada data valid.")
        return pd.DataFrame()

    df = pd.DataFrame(results)

    # ── APPLY FILTERS ────────────────────────────────────────────
    mask = (
        (df["range_ticks"] < cfg["max_ticks"])  &
        (df["vol_pct"]     < cfg["max_vol_pct"]) &
        (df["within_pct"]  == True)
    )
    if cfg["require_above_ma100"]:
        mask &= (df["above_ma100"] == True)
    if cfg["volume_filter"]:
        mask &= (df["volume"] >= cfg["min_volume"])

    screened = df[mask].sort_values("range_ticks").reset_index(drop=True)
    return screened


# ══════════════════════════════════════════════════════════════════
# 6. OUTPUT
# ══════════════════════════════════════════════════════════════════
def print_results(df: pd.DataFrame, cfg: dict):
    sep = "=" * 90
    print(f"\n{sep}")
    print(f"  HASIL SCREENING — MA KETAT (MA Menguncup)")
    print(f"  {datetime.today().strftime('%A, %d %B %Y')}")
    print(sep)

    if df.empty:
        print("\n  Tidak ada saham yang lolos kriteria hari ini.\n")
        return

    print(f"\n  Total saham lolos: {len(df)}\n")

    display_cols = ["ticker", "close", "range_ticks", "vol_pct",
                    "ma3", "ma5", "ma10", "ma20", "ma50",
                    "volume", "above_ma100", "tick_size"]

    display_df = df[display_cols].copy()
    display_df.columns = [
        "Ticker", "Close", "RangeTick", "VolPct%",
        "MA3", "MA5", "MA10", "MA20", "MA50",
        "Volume", "AboveMA100", "TickSz"
    ]
    display_df["Volume"] = display_df["Volume"].apply(lambda x: f"{x:,.0f}")

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.float_format", lambda x: f"{x:.1f}")
    print(display_df.to_string(index=False))

    print(f"\n{'-'*90}")
    print("  LEGENDA:")
    print("  RangeTick  = jarak max-min semua MA dalam satuan tick BEI (piecewise)")
    print("  VolPct%    = volatilitas rolling 10 hari (std daily return x100)")
    print("  AboveMA100 = Close >= MA100 (konteks bullish)")
    print(f"  Kriteria   : RangeTick < {cfg['max_ticks']}  &  VolPct < {cfg['max_vol_pct']}  &  Close +/-{cfg['max_deviation']}% semua MA")
    if cfg["volume_filter"]:
        print(f"  + Volume   >= {cfg['min_volume']:,} lembar (filter aktif)")
    print(f"{'-'*90}")

    out_file = f"ma_ketat_{datetime.today().strftime('%Y%m%d')}.csv"
    df.to_csv(out_file, index=False)
    print(f"\n  Hasil disimpan ke: {out_file}\n")


# ══════════════════════════════════════════════════════════════════
# 7. RUN
# ══════════════════════════════════════════════════════════════════
print("=" * 54)
print("  IHSG MA KETAT SCREENER")
print("  Rumus: range_ticks < 6 & vol_pct < 3.8")
print("  Source: @ikhwanuddin (X / Twitter, Mar 2026)")
print("=" * 54)
print(f"\n  Saham di-screen  : {len(CONFIG['custom_tickers']) or len(DEFAULT_TICKERS)}")
print(f"  Max range ticks  : {CONFIG['max_ticks']}")
print(f"  Max vol_pct      : {CONFIG['max_vol_pct']}%")
print(f"  Max dev dari MA  : {CONFIG['max_deviation']}%")
print(f"  Filter volume    : {'YA (>= ' + str(CONFIG['min_volume']) + ')' if CONFIG['volume_filter'] else 'TIDAK'}")
print(f"  Filter MA100     : {'YA (Close >= MA100)' if CONFIG['require_above_ma100'] else 'TIDAK'}")

result = run_screener(CONFIG)
print_results(result, CONFIG)

  IHSG MA KETAT SCREENER
  Rumus: range_ticks < 6 & vol_pct < 3.8
  Source: @ikhwanuddin (X / Twitter, Mar 2026)

  Saham di-screen  : 90
  Max range ticks  : 6.0
  Max vol_pct      : 3.8%
  Max dev dari MA  : 5.0%
  Filter volume    : TIDAK
  Filter MA100     : YA (Close >= MA100)

📥 Mengunduh data 90 saham dari Yahoo Finance...


[*********************100%***********************]  90 of 90 completed

🔍 Memproses indikator per saham...
⚠️  Tidak ada data valid.

  HASIL SCREENING — MA KETAT (MA Menguncup)
  Tuesday, 03 March 2026

  Tidak ada saham yang lolos kriteria hari ini.

